# 🎯 DipRadar — Bootstrap ML (Colab)

Corre cada célula **uma de cada vez**, por ordem.
O Parquet acumula no Google Drive — podes fechar o Colab entre sessões sem perder progresso.

| Passo | O que faz | Tempo estimado |
|-------|-----------|----------------|
| Batch 0–200 | Macro bulk fetch (3 pedidos) + backfill preços + features | ~25 min |
| Batch 200–400 | idem (macro já em cache no Drive) | ~25 min |
| Batch 400–600 | idem | ~25 min |
| Batch 600–800 | idem | ~25 min |
| Batch 800–fim | idem | ~10 min |
| Treino XGBoost + KNN | train_model.py | ~3 min |
| **Total** | | **~2 h 15 min** |

> ⚠️ **Cada sessão nova do Colab**: volta a correr as células 1, 2, 3 e 4 antes de continuar.

> ℹ️ O download macro (VIX, SPY, T10Y2Y) é feito automaticamente no **primeiro batch de cada sessão** — não há passo separado.

## 1 · Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive montado')

## 2 · Instalar dependências

In [ ]:
%pip install -q yfinance scikit-learn pyarrow fastparquet xgboost shap pandas-datareader
print('✅ Dependências instaladas')

## 3 · Clonar / actualizar repositório

In [ ]:
import os

REPO_DIR = '/content/DipRadar'

if os.path.exists(REPO_DIR):
    %cd $REPO_DIR
    !git pull --quiet
    print('✅ Repositório actualizado')
else:
    !git clone --quiet https://github.com/romeurf/DipRadar.git $REPO_DIR
    %cd $REPO_DIR
    print('✅ Repositório clonado')

DRIVE_DIR = '/content/drive/MyDrive/DipRadar'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'📁 Drive dir: {DRIVE_DIR}')

## 4 · Configurar API Key (Tiingo)

> Guarda a chave em: **Colab → Secrets** (ícone da chave 🔑) → Nome: `TIINGO_API_KEY`

In [ ]:
import os
from google.colab import userdata

try:
    os.environ['TIINGO_API_KEY'] = userdata.get('TIINGO_API_KEY')
    print('✅ TIINGO_API_KEY carregada dos Secrets do Colab')
except Exception:
    import getpass
    os.environ['TIINGO_API_KEY'] = getpass.getpass('🔑 Cola a tua TIINGO_API_KEY: ')
    print('✅ TIINGO_API_KEY definida via input')

## 5 · Batch 1 — tickers 0–200

> O `build_global_macro_df()` corre automaticamente no início deste batch (3 pedidos de rede, ~3 s).  
> Se a sessão expirar a meio, volta a correr — os tickers já processados são saltados.

In [ ]:
!python bootstrap_ml.py \
    --layer price \
    --slice 0 200 \
    --drive-dir /content/drive/MyDrive/DipRadar

## 6 · Batch 2 — tickers 200–400

In [ ]:
!python bootstrap_ml.py \
    --layer price \
    --slice 200 400 \
    --drive-dir /content/drive/MyDrive/DipRadar

## 7 · Batch 3 — tickers 400–600

In [ ]:
!python bootstrap_ml.py \
    --layer price \
    --slice 400 600 \
    --drive-dir /content/drive/MyDrive/DipRadar

## 8 · Batch 4 — tickers 600–800

In [ ]:
!python bootstrap_ml.py \
    --layer price \
    --slice 600 800 \
    --drive-dir /content/drive/MyDrive/DipRadar

## 9 · Batch 5 — tickers 800 até ao fim

In [ ]:
!python bootstrap_ml.py \
    --layer price \
    --slice 800 999 \
    --drive-dir /content/drive/MyDrive/DipRadar

## 10 · Treino XGBoost + KNN + Platt Scaling

> Só depois de todos os batches estarem completos.  
> Usa o `train_model.py` dedicado — **não** o bootstrap. Garante que o contrato de features é lido de `ml_features.py` (fonte única de verdade).

In [ ]:
!python train_model.py \
    --parquet /content/drive/MyDrive/DipRadar/ml_training_price.parquet \
    --output-dir /content/drive/MyDrive/DipRadar

## 11 · Validar contrato de features (anti Training-Serving Skew)

> Confirma que o `.pkl` treinado e o `ml_features.py` em produção têm **exactamente as mesmas features**, pela mesma ordem.  
> **Não saltes este passo** — é o guarda que impède um crash silencioso em produção.

In [ ]:
import pickle, pathlib
from ml_features import FEATURE_COLUMNS, N_FEATURES

pkl_path = pathlib.Path('/content/drive/MyDrive/DipRadar/dip_model_price.pkl')
assert pkl_path.exists(), f'❌ Modelo não encontrado em {pkl_path}'

with open(pkl_path, 'rb') as f:
    bundle = pickle.load(f)

model_features = list(bundle['model'].feature_names_in_)
code_features  = list(FEATURE_COLUMNS)

if model_features == code_features:
    print(f'✅ Contrato OK — {N_FEATURES} features idênticas entre modelo e ml_features.py')
    for i, f in enumerate(model_features):
        print(f'  {i+1:>2}. {f}')
else:
    missing = set(code_features) - set(model_features)
    extra   = set(model_features) - set(code_features)
    print('❌ MISMATCH DETECTADO — Training-Serving Skew activo!')
    if missing: print(f'  Faltam no modelo : {missing}')
    if extra:   print(f'  Extra no modelo  : {extra}')
    raise AssertionError('Corrige antes de fazer deploy.')

## 12 · Verificar todos os ficheiros gerados

In [ ]:
import os, pathlib, pandas as pd

drive_dir = pathlib.Path('/content/drive/MyDrive/DipRadar')
print('📁 Ficheiros no Drive:')
for f in sorted(drive_dir.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<40} {size_kb:>8.1f} KB')

pq = drive_dir / 'ml_training_price.parquet'
if pq.exists():
    df = pd.read_parquet(pq)
    print(f'\n📊 Parquet: {len(df):,} alertas | {df["symbol"].nunique()} tickers únicos')
    print('\nDistribuição outcome_label:')
    print(df['outcome_label'].value_counts().to_string())
else:
    print('⚠️  Parquet ainda não existe')

## 13 · Deploy do modelo para o Railway

### Opção A — Railway Volume (recomendado)
O Railway tem um Volume montado em `/data`.
Descarrega o `.pkl` do Drive para a tua máquina local e faz upload via Railway CLI:
```bash
railway volume cp ./dip_model_price.pkl /data/dip_model_price.pkl
railway volume cp ./dip_knn_price.pkl   /data/dip_knn_price.pkl
```

### Opção B — Commit directo (só se < 50 MB por ficheiro)
```bash
git add dip_model_price.pkl dip_knn_price.pkl
git commit -m 'feat: add trained ML models v1'
git push
```